# 10장. 순서가 있는 자료: 순환 신경망

시간별 전력수요에서 다음 한 시간과 다음 24시간을 예측한다. 기준선, 순환 신경망, LSTM과 GRU, 1차원 합성곱을 같은 자료로 비교한다.

**준비:** 전력수요 파일 두 개(`kpx_2023.csv`, `kpx_2025.csv`)를 왼쪽 파일 창에 올린 뒤 위에서부터 차례로 실행한다. 받는 곳은 저장소의 `README.md`에 있다. 끝까지 실행하는 데 15분쯤 걸린다.

## 10.1 기준선 만들기: 한 시간 전, 하루 전, 일주일 전의 값

In [ ]:
import glob, time
import numpy as np
import pandas as pd
import torch
from torch import nn

torch.set_num_threads(2)
files = sorted(glob.glob("kpx_*.csv"))
wide = pd.concat([pd.read_csv(f, encoding="cp949") for f in files])
wide = wide.dropna(subset=["날짜"])
long = wide.melt("날짜", var_name="시", value_name="MWh")
hours = pd.to_timedelta(long["시"].str.replace("시", "").astype(int), unit="h")
long["시각"] = pd.to_datetime(long["날짜"]) + hours
power = long.set_index("시각")["MWh"].sort_index() / 1000        # GW

year23 = power.loc["2023-01-01 01":"2024-01-01 00"]
year25 = power.loc["2025-01-01 01":"2026-01-01 00"]
print(files, wide.shape)
print(len(year23), len(year25), int(power.isna().sum()))
print(year23.iloc[:8].round(1).tolist())
for name, year in [("2023년", year23), ("2025년", year25)]:
    print(name, f"평균 {year.mean():.2f}  표준편차 {year.std():.2f}  "
                f"최소 {year.min():.2f}  최대 {year.max():.2f}")

In [ ]:
WEEK = 168

def windows(series):
    """목표(어느 한 시간의 수요)마다 그 앞 168시간의 값을 한 행으로 만든다."""
    values = series.to_numpy()
    past = np.lib.stride_tricks.sliding_window_view(values, WEEK)[:-1]
    target = values[WEEK:]
    return past.astype(np.float32), target.astype(np.float32), series.index[WEEK:]

P23, Y23, when23 = windows(year23)
P25, Y25, when25 = windows(year25)
held = when23.isocalendar().week.to_numpy() % 4 == 1    # 6장과 같은 검증 주
print(P23.shape, Y23.shape, P25.shape, int(held.sum()), int((~held).sum()))
print(when23[0], P23[0, -3:].round(1), Y23[0].round(1))
print(bool(P23[1, -1] == Y23[0]), bool(P23[24, -24] == Y23[0]),
      bool(P23[168, 0] == Y23[0]))

In [ ]:
def rmse(actual, predicted):
    return float(np.sqrt(((actual - predicted) ** 2).mean()))

guesses = {"2023년 학습 자료의 평균": np.full(len(Y25), Y23[~held].mean()),
           "한 시간 전의 값": P25[:, -1],
           "하루 전 같은 시각의 값": P25[:, -24],
           "일주일 전 같은 시각의 값": P25[:, 0]}
print("2025년 8,592시간, 다음 한 시간의 수요를 예측할 때의 RMSE(GW)")
for name, guess in guesses.items():
    print(f"{rmse(Y25, guess):>6.2f}  {name}")

In [ ]:
def with_one(past, L):
    """앞 L시간의 값에 절편을 위한 1의 열을 붙인다."""
    return np.column_stack([past[:, -L:], np.ones(len(past))]).astype(np.float64)

def linear(L):
    """앞 L시간의 값으로 다음 값을 맞히는 선형 회귀. 계수와 두 RMSE를 돌려준다."""
    coef = np.linalg.lstsq(with_one(P23[~held], L), Y23[~held], rcond=None)[0]
    return (coef, rmse(Y23[held], with_one(P23[held], L) @ coef),
            rmse(Y25, with_one(P25, L) @ coef))

print("입력 구간    검증    2025년")
for L in [1, 24, 48, 168]:
    coef, on_valid, on_2025 = linear(L)
    print(f"{L:>4}시간   {on_valid:.3f}   {on_2025:.3f}")
weights = coef[:-1]                                    # 168시간 모델의 계수(절편 제외)
largest = np.argsort(-np.abs(weights))[:6]
print([(int(WEEK - k), round(float(weights[k]), 2)) for k in largest])

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

print("그레이디언트 부스팅   검증    2025년")
for L in [24, 168]:
    lap = time.perf_counter()
    model = HistGradientBoostingRegressor(random_state=0)
    model.fit(P23[~held][:, -L:], Y23[~held])
    on_valid = rmse(Y23[held], model.predict(P23[held][:, -L:]))
    on_2025 = rmse(Y25, model.predict(P25[:, -L:]))
    print(f"{L:>4}시간          {on_valid:.3f}   {on_2025:.3f}   "
          f"{time.perf_counter() - lap:.1f}초")

## 10.2 순환 신경망을 넘파이로 만들기: 은닉 상태와 시간 방향 역전파

In [ ]:
def rnn_forward(x, W_x, W_h, b):
    """x: (걸음 수, 입력 수). 은닉 상태를 (걸음 수 + 1, 은닉 수)로 돌려준다."""
    h = np.zeros((len(x) + 1, len(b)))                 # h[0]은 처음의 은닉 상태(0)
    for t in range(len(x)):
        h[t + 1] = np.tanh(W_x @ x[t] + W_h @ h[t] + b)
    return h

x = np.array([[1.0], [0.5], [-1.0]])                   # 세 걸음, 걸음마다 숫자 하나
W_x = np.array([[0.5], [-1.0]])                        # (은닉 2, 입력 1)
W_h = np.array([[0.2, -0.5], [0.4, 0.1]])              # (은닉 2, 은닉 2)
b = np.array([0.0, 0.1])
h = rnn_forward(x, W_x, W_h, b)
print(h.round(4))

In [ ]:
layer = nn.RNN(1, 2, batch_first=True).double()
with torch.no_grad():
    layer.weight_ih_l0.copy_(torch.tensor(W_x))
    layer.weight_hh_l0.copy_(torch.tensor(W_h))
    layer.bias_ih_l0.copy_(torch.tensor(b)); layer.bias_hh_l0.zero_()
states, last = layer(torch.tensor(x)[None])            # 입력은 (배치, 걸음 수, 입력 수)
print(states.shape, last.shape)
print(float(np.abs(states[0].detach().numpy() - h[1:]).max()))
shapes = [tuple(p.shape) for p in layer.parameters()]
print(shapes, sum(p.numel() for p in layer.parameters()))

In [ ]:
v, y = np.array([1.0, -1.0]), 0.5                      # 출력 가중치와 정답

def loss_of(W_x, W_h, b):
    return float((v @ rnn_forward(x, W_x, W_h, b)[-1] - y) ** 2)

def bptt(W_x, W_h, b):
    """마지막 걸음에서 첫 걸음으로 거슬러 가며 기울기를 더한다."""
    h = rnn_forward(x, W_x, W_h, b)
    g_x, g_h, g_b = np.zeros_like(W_x), np.zeros_like(W_h), np.zeros_like(b)
    dh = 2 * (v @ h[-1] - y) * v                   # 손실을 마지막 은닉 상태로 미분한 값
    sizes = []
    for t in range(len(x), 0, -1):
        sizes.append(float(np.linalg.norm(dh)))
        da = dh * (1 - h[t] ** 2)                      # tanh를 지난다
        g_x += np.outer(da, x[t - 1]); g_h += np.outer(da, h[t - 1]); g_b += da
        dh = W_h.T @ da                                # 한 걸음 앞의 은닉 상태로 넘긴다
    return g_x, g_h, g_b, sizes

g_x, g_h, g_b, sizes = bptt(W_x, W_h, b)
print(round(loss_of(W_x, W_h, b), 4), np.round(sizes, 4))
print(g_h.round(4))

eps, numeric = 1e-6, np.zeros_like(W_h)
for i in range(2):
    for j in range(2):
        step = np.zeros_like(W_h); step[i, j] = eps
        up, down = loss_of(W_x, W_h + step, b), loss_of(W_x, W_h - step, b)
        numeric[i, j] = (up - down) / (2 * eps)
print(float(np.abs(numeric - g_h).max()))

layer.zero_grad()
out = torch.tensor(v) @ layer(torch.tensor(x)[None])[0][0, -1]
((out - y) ** 2).backward()
print(float(np.abs(layer.weight_hh_l0.grad.numpy() - g_h).max()),
      float(np.abs(layer.weight_ih_l0.grad.numpy() - g_x).max()))

In [ ]:
MEAN, STD = float(P23[~held].mean()), float(P23[~held].std())

class SeqNet(nn.Module):
    """순환 층 하나와 선형층 하나. 마지막 걸음의 은닉 상태로 outputs개의 값을 낸다."""
    def __init__(self, kind, hidden=32, outputs=1):
        super().__init__()
        layer = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[kind]
        self.recurrent = layer(1, hidden, batch_first=True)
        self.head = nn.Linear(hidden, outputs)

    def forward(self, x):                              # x: (배치, 걸음 수)
        states, _ = self.recurrent(x[:, :, None])      # (배치, 걸음 수, 은닉 수)
        return self.head(states[:, -1])                # (배치, outputs)

def scale(values, norm=None):
    mean, std = norm or (MEAN, STD)
    return torch.tensor((np.asarray(values) - mean) / std, dtype=torch.float32)

def fit(net, past, target, lr, seed=0, epochs=30, norm=None, clip=None, watch=None):
    """Adam, 배치 64. 학습률은 lr에서 시작해 코사인 모양으로 0까지 줄인다.
    clip: 기울기의 크기가 이 값을 넘으면 이 값으로 줄인다.
    watch: 걸음마다 기울기의 크기를 적어 둘 목록."""
    x, z = scale(past, norm), scale(target, norm)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    steps = epochs * -(-len(x) // 64)
    schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, steps)
    shuffler = torch.Generator().manual_seed(seed)
    for epoch in range(epochs):
        net.train()
        mixed = torch.randperm(len(x), generator=shuffler)
        for start in range(0, len(x), 64):
            rows = mixed[start:start + 64]
            optimizer.zero_grad()
            ((net(x[rows]) - z[rows]) ** 2).mean().backward()
            size = nn.utils.clip_grad_norm_(net.parameters(), clip or float("inf"))
            if watch is not None:
                watch.append(float(size))              # 줄이기 전의 크기
            optimizer.step(); schedule.step()
    return net

def predict(net, past, norm=None):
    """예측을 GW 단위로 돌려준다. (개수, 출력 수)"""
    mean, std = norm or (MEAN, STD)
    net.eval()
    with torch.no_grad():
        x = scale(past, norm)
        parts = [net(x[k:k + 2000]) for k in range(0, len(x), 2000)]
    return torch.cat(parts).numpy() * std + mean

ONE = (P23[~held], Y23[~held, None], P23[held], Y23[held, None], P25, Y25[:, None])
count = lambda net: sum(p.numel() for p in net.parameters())
scores, nets, took = {}, {}, {}

def trial(name, make_net, L, lr, data=ONE, seeds=5, clip=None):
    """시드를 바꿔 학습시키고 검증·2025년 RMSE를 scores[name]에 적는다."""
    tr_x, tr_y, va_x, va_y, te_x, te_y = data
    lap, rows, nets[name] = time.perf_counter(), [], []
    for seed in range(seeds):
        torch.manual_seed(seed)
        net = fit(make_net(), tr_x[:, -L:], tr_y, lr, seed, clip=clip)
        rows.append((rmse(va_y, predict(net, va_x[:, -L:])),
                     rmse(te_y, predict(net, te_x[:, -L:]))))
        nets[name].append(net)
    scores[name], took[name] = np.array(rows), time.perf_counter() - lap

def pad(text, width):
    wide = sum(ord(ch) > 0x1100 for ch in text)
    return text + " " * max(width - len(text) - wide, 1)

def show(*names):
    print(pad("모델, 입력 구간", 26) + "파라미터   검증    2025년 (범위)          시간")
    for name in names:
        s = scores[name]
        span = f"({s[:, 1].min():.3f}~{s[:, 1].max():.3f})"
        size, mid = count(nets[name][0]), np.median(s, axis=0)
        print(pad(name, 26) + f"{size:>7,}   {mid[0]:.3f}   {mid[1]:.3f} {span}  "
              f"{took[name]:>4.0f}초")

In [ ]:
print("학습률     RNN 검증   LSTM 검증  (시드 0, 입력 24시간)")
for lr in [0.001, 0.003, 0.01, 0.03, 0.1]:
    row = []
    for kind in ["RNN", "LSTM"]:
        torch.manual_seed(0)
        net = fit(SeqNet(kind), P23[~held][:, -24:], Y23[~held, None], lr)
        row.append(rmse(Y23[held, None], predict(net, P23[held][:, -24:])))
    print(f"{lr:<8}   {row[0]:.3f}      {row[1]:.3f}")

In [ ]:
LR = 0.03
trial("RNN, 24시간", lambda: SeqNet("RNN"), 24, LR)
show("RNN, 24시간")
print(scores["RNN, 24시간"][:, 1].round(3))

In [ ]:
print("시드   기울기 크기의 중앙값   최댓값   1보다 큰 걸음의 비율  (RNN, 24시간)")
for seed in [0, 4]:
    seen = []
    torch.manual_seed(seed)
    fit(SeqNet("RNN"), P23[~held][:, -24:], Y23[~held, None], LR, seed, watch=seen)
    sizes_seen = np.array(seen)
    over = (sizes_seen > 1).mean() * 100
    middle_size, largest_size = np.median(sizes_seen), sizes_seen.max()
    print(f"{seed:>3}    {middle_size:.3f}               {largest_size:>6.1f}"
          f"   {over:.1f}%")
trial("RNN + 기울기 자르기, 24시간", lambda: SeqNet("RNN"), 24, LR, clip=1.0)
show("RNN, 24시간", "RNN + 기울기 자르기, 24시간")
print(scores["RNN + 기울기 자르기, 24시간"][:, 1].round(3))

In [ ]:
shuffled = np.random.default_rng(0).permutation(len(Y23))
random_held = np.zeros(len(Y23), dtype=bool)
random_held[shuffled[:held.sum()]] = True               # 같은 수를 무작위로 뗀다
both_years = np.concatenate([year23, year25])           # 2025년까지 넣은 값
whole = (float(both_years.mean()), float(both_years.std()))
print(round(MEAN, 2), round(STD, 2), round(whole[0], 2), round(whole[1], 2))
print("검증 자료를 뗀 방법, 표준화        검증    2025년  (LSTM, 24시간, 시드 0)")
for name, mask, norm in [("주 단위, 학습 자료로 표준화", held, None),
                         ("무작위, 학습 자료로 표준화", random_held, None),
                         ("주 단위, 2025년까지 넣어 표준화", held, whole)]:
    torch.manual_seed(0)
    net = fit(SeqNet("LSTM"), P23[~mask][:, -24:], Y23[~mask, None], LR, norm=norm)
    on_valid = rmse(Y23[mask, None], predict(net, P23[mask][:, -24:], norm))
    on_2025 = rmse(Y25[:, None], predict(net, P25[:, -24:], norm))
    print(pad(name, 34) + f"{on_valid:.3f}   {on_2025:.3f}")

## 10.3 입력 구간을 늘릴 때의 기울기 소실과 LSTM·GRU

In [ ]:
trial("RNN, 168시간", lambda: SeqNet("RNN"), 168, LR)
show("RNN, 24시간", "RNN, 168시간")
print(scores["RNN, 168시간"][:, 1].round(3))

In [ ]:
def reach(name):
    """예측을 입력의 각 시각으로 미분한 값의 크기. 검증 자료 500개의 평균이다."""
    L = int(name.split(", ")[1].replace("시간", ""))
    rows = []
    for net in nets[name]:
        x = scale(P23[held][:500, -L:]).requires_grad_(True)
        net.eval()
        net(x).sum().backward()
        rows.append(x.grad.abs().mean(dim=0).numpy()[::-1])      # 0번 칸이 한 시간 전
    return np.array(rows)

far = {"RNN, 168시간": reach("RNN, 168시간")}
def by_lag(name, label):
    """시드 5개의 중앙값을 몇 시간 전인지에 따라 적는다."""
    mid = np.median(far[name], axis=0)
    picked = "  ".join(f"{mid[k - 1]:.1e}" for k in [1, 2, 6, 12, 24, 48, 168])
    print(pad(label, 11) + picked)

print("몇 시간 전     1        2        6       12       24       48      168")
by_lag("RNN, 168시간", "RNN")
print(far["RNN, 168시간"][:, 23])

In [ ]:
def lstm_step(x, h, c, W_x, W_h, b):
    """LSTM의 한 걸음. 가중치의 행은 입력, 망각, 후보, 출력의 순서로 쌓여 있다."""
    n = len(h)
    a = W_x @ x + W_h @ h + b                          # 길이 4n
    gate = lambda part: 1 / (1 + np.exp(-part))        # 시그모이드
    i, f, o = gate(a[:n]), gate(a[n:2 * n]), gate(a[3 * n:])
    g = np.tanh(a[2 * n:3 * n])                        # 새로 적을 후보 값
    c_new = f * c + i * g                          # 셀 상태: 일부를 남기고 일부를 더함
    return o * np.tanh(c_new), c_new, (i, f, g, o)

torch.manual_seed(0)
cell = nn.LSTM(1, 2, batch_first=True).double()
names = ["weight_ih_l0", "weight_hh_l0", "bias_ih_l0", "bias_hh_l0"]
A_x, A_h, b_1, b_2 = [getattr(cell, name).detach().numpy() for name in names]
print(A_x.shape, A_h.shape, b_1.shape, sum(p.numel() for p in cell.parameters()))
h_t, c_t = np.zeros(2), np.zeros(2)
for t in range(len(x)):
    h_t, c_t, gates = lstm_step(x[t], h_t, c_t, A_x, A_h, b_1 + b_2)
    if t == 0:
        for label, part in zip(["i", "f", "g", "o"], gates):
            print(label, part.round(4))
        print("c", c_t.round(4), " h", h_t.round(4))
states, (h_last, c_last) = cell(torch.tensor(x)[None])
print(float(np.abs(h_last[0, 0].detach().numpy() - h_t).max()),
      float(np.abs(c_last[0, 0].detach().numpy() - c_t).max()))

In [ ]:
trial("LSTM, 24시간", lambda: SeqNet("LSTM"), 24, LR)
trial("GRU, 24시간", lambda: SeqNet("GRU"), 24, LR)
trial("LSTM, 168시간", lambda: SeqNet("LSTM"), 168, LR)
show("RNN, 24시간", "LSTM, 24시간", "GRU, 24시간", "RNN, 168시간", "LSTM, 168시간")
far["LSTM, 168시간"] = reach("LSTM, 168시간")
print("몇 시간 전     1        2        6       12       24       48      168")
by_lag("RNN, 168시간", "RNN")
by_lag("LSTM, 168시간", "LSTM")

## 10.4 1차원 합성곱, 펴서 넣는 신경망과 비교하기

In [ ]:
class ConvNet(nn.Module):
    """1차원 합성곱 층 두 개(커널 3, 채널 32) 뒤에 펴서 선형층에 넣는다."""
    def __init__(self, L):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(32, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.Flatten(), nn.Linear(32 * L, 1))

    def forward(self, x):                              # x: (배치, 걸음 수)
        return self.layers(x[:, None])                 # 채널 축을 넣음: (배치, 1, 걸음)

def make_mlp(L):
    return nn.Sequential(nn.Linear(L, 128), nn.ReLU(), nn.Linear(128, 1))

sample = torch.zeros(2, 24)
print(nn.Conv1d(1, 32, 3, padding=1)(sample[:, None]).shape, ConvNet(24)(sample).shape)
print("학습률     합성곱 검증   펴서 넣기 검증  (시드 0, 입력 24시간)")
for lr in [0.003, 0.01, 0.03]:
    row = []
    for make_net in [lambda: ConvNet(24), lambda: make_mlp(24)]:
        torch.manual_seed(0)
        net = fit(make_net(), P23[~held][:, -24:], Y23[~held, None], lr)
        row.append(rmse(Y23[held, None], predict(net, P23[held][:, -24:])))
    print(f"{lr:<8}   {row[0]:.3f}         {row[1]:.3f}")

In [ ]:
for L in [24, 168]:
    trial(f"1차원 합성곱, {L}시간", lambda: ConvNet(L), L, 0.01)      # 검증에서 고른 학습률
    trial(f"펴서 넣는 신경망, {L}시간", lambda: make_mlp(L), L, 0.03)
show("RNN, 24시간", "LSTM, 24시간", "GRU, 24시간", "1차원 합성곱, 24시간",
     "펴서 넣는 신경망, 24시간")
show("RNN, 168시간", "LSTM, 168시간", "1차원 합성곱, 168시간",
     "펴서 넣는 신경망, 168시간")

## 10.5 24시간 앞까지 예측하기

In [ ]:
def spans(series):
    """예측을 시작하는 시각마다 앞 168시간(past)과 뒤 24시간(future)을 만든다."""
    values = series.to_numpy().astype(np.float32)
    both = np.lib.stride_tricks.sliding_window_view(values, WEEK + 24)
    return both[:, :WEEK], both[:, WEEK:], series.index[WEEK:len(values) - 23]

Q23, F23, start23 = spans(year23)
Q25, F25, start25 = spans(year25)
held_day = start23.isocalendar().week.to_numpy() % 4 == 1
DAY = (Q23[~held_day], F23[~held_day], Q23[held_day], F23[held_day], Q25, F25)
print(Q23.shape, F23.shape, Q25.shape, int(held_day.sum()))

def by_hour(actual, predicted):
    """몇 시간 앞인지에 따른 RMSE 24개."""
    return np.sqrt(((actual - predicted) ** 2).mean(axis=0))

ahead = {"한 시간 전의 값을 24시간 유지": np.repeat(Q25[:, -1:], 24, axis=1),
         "하루 전 같은 시각의 값": Q25[:, -24:],
         "일주일 전 같은 시각의 값": Q25[:, :24]}
for L in [24, 168]:
    future = F23[~held_day].astype(np.float64)         # 목표가 24열이다
    many = np.linalg.lstsq(with_one(Q23[~held_day], L), future, rcond=None)[0]
    ahead[f"선형 회귀, {L}시간"] = with_one(Q25, L) @ many

def errors(name):
    """몇 시간 앞인지에 따른 RMSE 24개와 전체 RMSE. 시드가 여럿이면 중앙값이다."""
    runs = ahead[name] if ahead[name].ndim == 3 else ahead[name][None]
    each = np.median([by_hour(F25, run) for run in runs], axis=0)
    return each, float(np.median([rmse(F25, run) for run in runs]))

def report(names):
    print(pad("2025년 RMSE(GW)", 32) + "  1시간   3시간   6시간  12시간  24시간   전체")
    for name in names:
        each, overall = errors(name)
        picked = "".join(f"{each[k - 1]:>7.2f}" for k in [1, 3, 6, 12, 24])
        print(pad(name, 32) + picked + f"{overall:>8.2f}")

report(list(ahead))

In [ ]:
def roll(net, past, L=24):
    """한 시간 앞 모델의 예측을 입력의 끝에 넣어 가며 24시간 앞까지 예측한다."""
    x, outs = scale(past[:, -L:]), []
    net.eval()
    with torch.no_grad():
        for _ in range(24):
            nxt = net(x)                               # (개수, 1)
            outs.append(nxt)
            x = torch.cat([x[:, 1:], nxt], dim=1)   # 가장 오래된 값을 빼고 예측을 넣음
    return torch.cat(outs, dim=1).numpy() * STD + MEAN

rolled = [roll(net, Q25) for net in nets["LSTM, 24시간"]]
ahead["LSTM, 예측을 다시 넣기"] = np.array(rolled)
print(ahead["LSTM, 예측을 다시 넣기"].shape)                  # (시드, 시작 시각, 24)
report(["LSTM, 예측을 다시 넣기"])
last_hour = [by_hour(F25, run)[23] for run in ahead["LSTM, 예측을 다시 넣기"]]
print(np.round(last_hour, 2))

In [ ]:
trial("LSTM 24개 출력, 24시간", lambda: SeqNet("LSTM", outputs=24), 24, LR, data=DAY)
trial("LSTM 24개 출력, 168시간", lambda: SeqNet("LSTM", outputs=24), 168, LR, data=DAY)
show("LSTM 24개 출력, 24시간", "LSTM 24개 출력, 168시간")
for name, L in [("LSTM 24개 출력, 24시간", 24), ("LSTM 24개 출력, 168시간", 168)]:
    ahead[name] = np.array([predict(net, Q25[:, -L:]) for net in nets[name]])
report(["LSTM, 예측을 다시 넣기", "LSTM 24개 출력, 24시간", "LSTM 24개 출력, 168시간",
        "선형 회귀, 168시간", "일주일 전 같은 시각의 값"])

In [ ]:
monday = pd.Timestamp("2025-07-07 01:00")              # 월요일 1시부터 24시간
spot = int(np.flatnonzero(start25 == monday)[0])
print(start25[spot], start25[spot].day_name(), F25[spot].round(1)[[0, 5, 11, 17, 23]])
for name in ["하루 전 같은 시각의 값", "일주일 전 같은 시각의 값", "선형 회귀, 168시간",
             "LSTM, 예측을 다시 넣기", "LSTM 24개 출력, 168시간"]:
    guess = ahead[name][0] if ahead[name].ndim == 3 else ahead[name]   # 신경망은 시드 0
    print(pad(name, 28) + f"{rmse(F25[spot], guess[spot]):.2f}")

In [ ]:
import sys, platform, sklearn

print(sys.version.split()[0], np.__version__, pd.__version__, torch.__version__,
      sklearn.__version__)
print(platform.machine(), torch.get_default_dtype(), torch.get_num_threads())

In [ ]:
!pip -q install koreanize-matplotlib

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
from matplotlib.colors import to_rgba
from matplotlib.ticker import FuncFormatter

NAVY, BRICK, TEAL, GRAY = "#16324f", "#b5532a", "#2a7f73", "#9aa1ab"
BOX = {"fc": "white", "ec": "none", "alpha": 0.9, "pad": 1.5}


def frame(ax, title=None, color=NAVY, grid="y"):
    """그래프마다 반복하는 꾸밈: 제목, 보조선, 테두리 정리."""
    if title:
        ax.set_title(title, fontsize=9, loc="left", color=color, weight="bold")
    ax.grid(axis=grid, color="#e3e7ec", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=9)
    ax.spines[["top", "right"]].set_visible(False)


def plain_ticks(axis):
    """로그 축의 눈금을 10의 거듭제곱 대신 보통 숫자로 적는다."""
    axis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:,.0f}" if val >= 1 else f"{val:.7f}".rstrip("0")))

In [ ]:
# 그림 10-1
fig, axes = plt.subplots(2, 1, figsize=(7, 7.4))
ax = axes[0]
two_weeks = year23.loc["2023-01-09 01":"2023-01-23 00"]            # 월요일 1시부터 14일
spot_hours = np.arange(len(two_weeks))
for day in range(14):
    if day % 7 >= 5:                                               # 토요일과 일요일
        ax.axvspan(day * 24, day * 24 + 24, color="#f1f2f4", lw=0)
ax.plot(spot_hours, two_weeks.to_numpy(), color=NAVY, lw=1.4)
goal = 7 * 24 + 2 * 24 + 13                                        # 둘째 주 수요일 14시
goal_value = two_weeks.iloc[goal]
for back, word, lift in [(1, "한 시간 전", 0), (24, "하루 전 같은 시각", 0), (168, "일주일 전 같은 시각", 0)]:
    ax.scatter(goal - back, two_weeks.iloc[goal - back], s=34, color=BRICK, zorder=6)
ax.scatter(goal, goal_value, s=46, color=TEAL, zorder=7)
ax.annotate("맞혀야 하는 값\n(둘째 주 수요일 14시)", xy=(goal, goal_value), xytext=(goal + 30, 97.5), fontsize=9,
            color=TEAL, ha="center", va="center", linespacing=1.4, arrowprops=dict(arrowstyle="->", color=TEAL, lw=0.9))
ax.annotate("하루 전 같은 시각", xy=(goal - 24, two_weeks.iloc[goal - 24]), xytext=(goal - 52, 97.5), fontsize=9,
            color=BRICK, ha="center", va="center", arrowprops=dict(arrowstyle="->", color=BRICK, lw=0.9))
ax.annotate("일주일 전 같은 시각", xy=(goal - 168, two_weeks.iloc[goal - 168]), xytext=(goal - 168, 97.5),
            fontsize=9, color=BRICK, ha="center", va="center",
            arrowprops=dict(arrowstyle="->", color=BRICK, lw=0.9))
ax.set_xticks(np.arange(0, 14 * 24 + 1, 24) + 12, minor=True)
ax.set_xticks(np.arange(0, 14 * 24 + 1, 24))
ax.set_xticklabels([])
ax.set_xticks(np.arange(14) * 24 + 12, minor=True)
ax.set_xticklabels(list("월화수목금토일") * 2, minor=True, fontsize=9)
ax.tick_params(axis="x", which="minor", length=0)
ax.annotate("한 시간 전", xy=(goal - 1, two_weeks.iloc[goal - 1]), xytext=(goal - 2, 56), fontsize=9,
            color=BRICK, ha="center", va="center", bbox=BOX, arrowprops=dict(arrowstyle="->", color=BRICK, lw=0.9))
ax.set_xlim(0, 14 * 24); ax.set_ylim(47, 102)
ax.set_ylabel("전력수요 (GW)", fontsize=9)
frame(ax, title="2023년 1월 9일부터 2주일의 시간별 전력수요 (회색은 토요일과 일요일)")
ax = axes[1]
kind = np.where(year23.index.dayofweek < 5, "평일", np.where(year23.index.dayofweek == 5, "토요일", "일요일"))
hour_of_day = (year23.index - pd.Timedelta(hours=1)).hour + 1       # 1시~24시
shape = year23.groupby([kind, hour_of_day]).mean().unstack(0)
for name, col, dash in [("평일", NAVY, "-"), ("토요일", TEAL, "--"), ("일요일", BRICK, ":")]:
    ax.plot(shape.index, shape[name], color=col, ls=dash, lw=2.0, marker="o", ms=3,
            label=f"{name} (가장 높은 시각 {int(shape[name].idxmax())}시, {shape[name].max():.1f}GW)")
ax.set_xticks(range(1, 25, 2)); ax.set_xlim(0.5, 24.5); ax.set_ylim(45, 80)
ax.set_xlabel("시각 (1시는 0시부터 1시까지의 수요)", fontsize=9); ax.set_ylabel("전력수요의 평균 (GW)", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="lower right")
frame(ax, title="2023년 한 해의 시각별 평균: 요일에 따라 모양이 다르다")
fig.tight_layout(h_pad=1.6)
fig.savefig("fig_10_1.png", dpi=300)
plt.show()
print(two_weeks.index[goal], round(float(goal_value), 1), [round(float(two_weeks.iloc[goal - back]), 1) for back in (1, 24, 168)])
print({name: (int(shape[name].idxmax()), round(float(shape[name].max()), 1), int(shape[name].idxmin()),
              round(float(shape[name].min()), 1)) for name in ["평일", "토요일", "일요일"]})

In [ ]:
# 그림 10-3
fig, ax = plt.subplots(figsize=(7, 4.6))
lags = WEEK - np.arange(WEEK)                                       # weights[k]는 lags[k]시간 전의 값에 곱한다
shade = np.where(weights > 0, NAVY, BRICK)
ax.bar(lags, weights, width=0.9, color=shade)
for k in largest:
    side = 1 if weights[k] > 0 else -1
    ax.text(lags[k] + {5: 16, 2: -4, 24: -6, 25: 6}.get(int(lags[k]), 0), weights[k] + side * 0.1 + (0.5 if lags[k] == 5 else 0),
            f"{lags[k]}시간 전\n{weights[k]:+.2f}".replace("-", "$-$"), fontsize=9,
            color=NAVY if side > 0 else BRICK, ha="center", va="bottom" if side > 0 else "top", linespacing=1.3)
for mark in [24, 48, 72, 96, 120, 144, 168]:
    ax.axvline(mark, color="#d5dae1", lw=0.7, zorder=0)
ax.axhline(0, color="#3a4350", lw=0.6)
ax.set_xticks([1, 24, 48, 72, 96, 120, 144, 168]); ax.set_xlim(172, -3); ax.set_ylim(-1.6, 2.4)
ax.set_yticks([-1, 0, 1, 2]); ax.set_yticklabels(["$-$1", "0", "1", "2"])
ax.set_xlabel("몇 시간 전의 값인가 (오른쪽이 최근)", fontsize=9); ax.set_ylabel("선형 회귀의 계수", fontsize=9)
frame(ax)
fig.tight_layout()
fig.savefig("fig_10_3.png", dpi=300)
plt.show()
print(round(float(np.abs(weights[:WEEK - 48]).max()), 2), int((np.abs(weights) > 0.1).sum()))

In [ ]:
# 그림 10-5
fig, axes = plt.subplots(2, 1, figsize=(7, 7.2), gridspec_kw={"height_ratios": [1.25, 1]})
ax = axes[0]
ax.plot(np.arange(1, len(sizes_seen) + 1), sizes_seen, color=BRICK, lw=0.5)
ax.axhline(1, color=NAVY, lw=1.0, ls="--")
ax.text(len(sizes_seen) * 0.995, 1.25, "기울기 자르기의 기준 1", fontsize=9, color=NAVY, ha="right")
ax.set_yscale("log"); plain_ticks(ax.yaxis)
ax.set_xlim(0, len(sizes_seen)); ax.set_xlabel("걸음 (30에폭, 3,090걸음)", fontsize=9)
ax.xaxis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:,.0f}"))
ax.set_ylabel("기울기의 크기 (로그 축)", fontsize=9)
frame(ax, title="RNN(시드 4)을 학습시키는 동안 걸음마다의 기울기 크기")
ax = axes[1]
pairs = [("RNN, 24시간", "기울기를 자르지 않음", BRICK), ("RNN + 기울기 자르기, 24시간", "크기 1에서 자름", TEAL)]
for row, (name, word, col) in enumerate(pairs):
    values = scores[name][:, 1]
    ax.scatter(values, np.full(5, row), s=46, color=col, zorder=5)
    for seed, value in enumerate(values):
        if value > 0.8:                                             # 따로 떨어진 시드만 번호를 적는다
            ax.text(value, row + 0.2, f"시드 {seed}: {value:.3f}", fontsize=9, color=col, ha="center")
    ax.plot([np.median(values)] * 2, [row - 0.28, row + 0.08], color=col, lw=2.2)
    ax.text(np.median(values), row - 0.36, f"중앙값 {np.median(values):.3f}", fontsize=9, color=col, ha="center",
            weight="bold")
ax.set_yticks([0, 1]); ax.set_yticklabels([word for _, word, _ in pairs], fontsize=9)
ax.set_ylim(1.6, -0.7); ax.set_xlim(0.5, 1.6)
ax.set_xlabel("2025년 RMSE (GW), 점 하나가 시드 하나", fontsize=9)
frame(ax, title="같은 RNN을 시드 5개로 학습시킨 결과", grid="x")
fig.tight_layout(h_pad=1.6)
fig.savefig("fig_10_5.png", dpi=300)
plt.show()
print(len(sizes_seen), round(float(sizes_seen.max()), 1))

In [ ]:
# 그림 10-7
fig, ax = plt.subplots(figsize=(7, 4.8))
for name, col, dash in [("RNN, 168시간", BRICK, "-"), ("LSTM, 168시간", NAVY, "--")]:
    for row in far[name]:
        ax.plot(np.arange(1, WEEK + 1), row, color=col, lw=0.5, alpha=0.35)
    mid = np.median(far[name], axis=0)
    ax.plot(np.arange(1, WEEK + 1), mid, color=col, ls=dash, lw=2.0,
            label=f"{name.split(',')[0]} (168시간 전의 값: {mid[-1]:.1e})")
for mark in [24, 48, 72, 96, 120, 144, 168]:
    ax.axvline(mark, color="#d5dae1", lw=0.7, zorder=0)
ax.set_yscale("log")
ax.yaxis.set_major_formatter(FuncFormatter(lambda val, _: f"$10^{{{int(round(np.log10(val)))}}}$"))
ax.set_xticks([1, 24, 48, 72, 96, 120, 144, 168]); ax.set_xlim(172, -3)
ax.set_xlabel("몇 시간 전의 입력인가 (오른쪽이 최근)", fontsize=9)
ax.set_ylabel("예측을 그 입력으로 미분한 값의 크기 (로그 축)", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="lower right", title="굵은 선: 시드 5개의 중앙값, 가는 선: 시드마다",
          title_fontsize=9)
frame(ax)
fig.tight_layout()
fig.savefig("fig_10_7.png", dpi=300)
plt.show()

In [ ]:
# 그림 10-8
rows_24 = ["RNN, 24시간", "RNN + 기울기 자르기, 24시간", "LSTM, 24시간", "GRU, 24시간",
           "1차원 합성곱, 24시간", "펴서 넣는 신경망, 24시간"]
rows_168 = ["RNN, 168시간", "LSTM, 168시간", "1차원 합성곱, 168시간", "펴서 넣는 신경망, 168시간"]
fig, ax = plt.subplots(figsize=(7, 6.2))
edge = 1.55                                                         # 이보다 큰 값은 오른쪽 끝에 적는다
spots, labels_here = [], []
for row, name in enumerate(rows_24 + rows_168):
    y = row + (0.8 if name in rows_168 else 0)
    col = NAVY if name in rows_24 else TEAL
    values = scores[name][:, 1]
    inside = values[values <= edge]
    ax.scatter(inside, np.full(len(inside), y), s=30, color=col, alpha=0.75, zorder=5)
    ax.plot([np.median(values)] * 2, [y - 0.3, y + 0.3], color=col, lw=2.4)
    ax.text(min(np.median(values), edge) + 0.02, y - 0.36, f"{np.median(values):.3f}", fontsize=9, color=col,
            weight="bold")
    for value in values[values > edge]:
        ax.annotate(f"{value:.2f}", xy=(edge + 0.04, y), xytext=(edge - 0.1, y + 0.02), fontsize=9, color=BRICK,
                    va="center", arrowprops=dict(arrowstyle="->", color=BRICK, lw=0.8))
    spots.append(y); labels_here.append(name.replace(", 24시간", "").replace(", 168시간", ""))
for L, dash in [(24, ":"), (168, "--")]:
    value = linear(L)[2]
    ax.axvline(value, color=BRICK, lw=1.0, ls=dash)
    ax.text(value + 0.01, -0.95, f"선형 회귀\n{L}시간 {value:.3f}", fontsize=9, color=BRICK, va="center", linespacing=1.3)
ax.set_yticks(spots); ax.set_yticklabels(labels_here, fontsize=9)
ax.text(0.405, -0.62, "입력 24시간", fontsize=9, color=NAVY, weight="bold", va="center")
ax.text(0.405, 6.2, "입력 168시간", fontsize=9, color=TEAL, weight="bold", va="center")
ax.set_ylim(10.4, -1.5); ax.set_xlim(0.4, edge + 0.08)
ax.set_xlabel("2025년 RMSE (GW), 다음 한 시간. 점은 시드마다, 굵은 세로선과 숫자는 중앙값", fontsize=9)
frame(ax, grid="x")
fig.tight_layout()
fig.savefig("fig_10_8.png", dpi=300)
plt.show()

In [ ]:
# 그림 10-10
fig, ax = plt.subplots(figsize=(7, 5.4))
styles = [("한 시간 전의 값을 24시간 유지", GRAY, ":", 1.4), ("하루 전 같은 시각의 값", GRAY, "--", 1.4),
          ("일주일 전 같은 시각의 값", GRAY, "-", 1.4), ("LSTM, 예측을 다시 넣기", BRICK, "-", 2.0),
          ("LSTM 24개 출력, 24시간", TEAL, "--", 2.0), ("LSTM 24개 출력, 168시간", TEAL, "-", 2.0),
          ("선형 회귀, 168시간", NAVY, "-", 2.2)]
for name, col, dash, width in styles:
    each, overall = errors(name)
    ax.plot(range(1, 25), each, color=col, ls=dash, lw=width, label=f"{name} (24시간 전체 {overall:.2f})")
ax.set_xticks([1, 3, 6, 9, 12, 15, 18, 21, 24]); ax.set_xlim(0.5, 24.5); ax.set_ylim(0, 15.5)
ax.set_xlabel("몇 시간 앞을 예측하는가", fontsize=9); ax.set_ylabel("2025년 RMSE (GW)", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="upper left", ncol=2, columnspacing=1.0)
frame(ax)
fig.tight_layout()
fig.savefig("fig_10_10.png", dpi=300)
plt.show()

In [ ]:
# 그림 10-11
fig, ax = plt.subplots(figsize=(7, 4.8))
before = np.arange(-47, 1)                                         # 예측을 시작하기 전 48시간
after = np.arange(1, 25)
ax.axvspan(0.5, 24.5, color="#f1f2f4", lw=0)
ax.plot(before, Q25[spot, -48:], color="#3a4350", lw=1.6)
ax.plot(after, F25[spot], color="#3a4350", lw=2.4, label="실제 수요")
shown = [("하루 전 같은 시각의 값", GRAY, "--"), ("선형 회귀, 168시간", NAVY, "-"),
         ("LSTM, 예측을 다시 넣기", BRICK, "-"), ("LSTM 24개 출력, 168시간", TEAL, "-")]
for name, col, dash in shown:
    guess = ahead[name][0] if ahead[name].ndim == 3 else ahead[name]
    ax.plot(after, guess[spot], color=col, ls=dash, lw=1.6,
            label=f"{name} (이날의 RMSE {rmse(F25[spot], guess[spot]):.2f})")
ax.axvline(0.5, color=BRICK, lw=0.9)
ax.text(0.0, 99.0, "예측하는 시점 ", fontsize=9, color=BRICK, ha="right", va="center")
ax.text(12.5, 99.0, "예측하는 24시간 (월요일)", fontsize=9, color="#3a4350", ha="center", va="center")
ax.text(-24, 99.0, "토요일과 일요일", fontsize=9, color="#3a4350", ha="center", va="center")
ax.set_xticks([-47, -36, -24, -12, 0, 6, 12, 18, 24]); ax.set_xlim(-47.5, 24.5); ax.set_ylim(50, 102)
ax.set_xticklabels(["$-$47", "$-$36", "$-$24", "$-$12", "0", "6", "12", "18", "24"])
ax.set_xlabel("예측하는 시점에서 몇 시간 뒤인가 (2025년 7월 7일 월요일 1시가 1)", fontsize=9)
ax.set_ylabel("전력수요 (GW)", fontsize=9)
ax.legend(frameon=False, fontsize=9, loc="upper left", bbox_to_anchor=(0.0, 0.93))
frame(ax)
fig.tight_layout()
fig.savefig("fig_10_11.png", dpi=300)
plt.show()
print(round(float(Q25[spot, -48:].max()), 1), round(float(F25[spot].max()), 1), int(F25[spot].argmax()) + 1)